In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
url = "http://ufcstats.com/fight-details/568ec6af4008355a"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
print(resp.status_code)

html = resp.text

200


In [3]:
soup = BeautifulSoup(html, "html.parser")

In [4]:
lutadores = soup.find_all("div", class_="b-fight-details__person")

for pessoa in lutadores:
    nome = pessoa.find("a", class_="b-link b-fight-details__person-link").text.strip()
    resultado = pessoa.find("i", class_="b-fight-details__person-status").text.strip()
    print(nome, resultado)

In [5]:
print(len(lutadores))

0


In [6]:
print(html[:500])

<!doctype html><html><head><meta charset="utf-8">
<title>Loading…</title><meta name="robots" content="noindex">
<style>body{font-family:sans-serif;color:#666;text-align:center;margin-top:25vh}</style>
</head><body>
<p>Checking your browser…</p>
<noscript>This site requires JavaScript.</noscript>
<script>
(function(){
var K=[0x428a2f98,0x71374491,0xb5c0fbcf,0xe9b5dba5,0x3956c25b,0x59f111f1,0x923f82a4,0xab1c5ed5,
0xd807aa98,0x12835b01,0x243185be,0x550c7dc3,0x72be5d74,0x80deb1fe,0x9bdc06a7,0xc19bf1


In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--headless=new")  # roda sem abrir janela visível
options.add_argument("--disable-blink-features=AutomationControlled")  # ajuda a não ser detectado como bot
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

In [8]:
import time

url = "http://ufcstats.com/fight-details/568ec6af4008355a"
driver.get(url)

time.sleep(5)  # espera a verificação/JS terminar de carregar

html = driver.page_source
print(html[:500])

<html class=" js flexbox canvas canvastext webgl no-touch geolocation postmessage no-websqldatabase indexeddb hashchange history draganddrop websockets rgba hsla multiplebgs backgroundsize borderimage borderradius boxshadow textshadow opacity cssanimations csscolumns cssgradients cssreflections csstransforms csstransforms3d csstransitions fontface generatedcontent video audio localstorage sessionstorage webworkers no-applicationcache svg inlinesvg smil svgclippaths" style=""><!--<![endif]--><hea


In [9]:
soup = BeautifulSoup(html, "html.parser")
lutadores = soup.find_all("div", class_="b-fight-details__person")
print(len(lutadores))

2


In [10]:
for pessoa in lutadores:
    nome = pessoa.find("a", class_="b-link b-fight-details__person-link").text.strip()
    resultado = pessoa.find("i", class_="b-fight-details__person-status").text.strip()
    print(nome, resultado)

Joshua Van W
Alexandre Pantoja L


In [11]:
info_bloco = soup.find("p", class_="b-fight-details__text")
itens = info_bloco.find_all("i", class_=["b-fight-details__text-item_first", "b-fight-details__text-item"])

for item in itens:
    label = item.find("i", class_="b-fight-details__label").text.strip()
    valor = item.get_text().replace(label, "").strip()
    print(label, "->", valor)

Method: -> Decision - Unanimous
Round: -> 5
Time: -> 5:00
Time format: -> 5 Rnd (5-5-5-5-5)
Referee: -> Herb Dean


In [13]:
from io import StringIO
import pandas as pd

tabelas = pd.read_html(StringIO(html))
print(len(tabelas))

4


In [14]:
for i, tabela in enumerate(tabelas):
    print(f"--- Tabela {i} ---")
    print(tabela.head())
    print()

--- Tabela 0 ---
                        Fighter    KD               Sig. str. Sig. str. %  \
0  Joshua Van Alexandre Pantoja  1  0  181 of 305  147 of 284    59%  51%   

               Total str.               Td      Td % Sub. att  Rev.  \
0  259 of 392  159 of 300  0 of 0  5 of 18  ---  27%     1  0  1  0   

         Ctrl  
0  2:13  9:11  

--- Tabela 1 ---
                        Fighter      KD           Sig. str. Sig. str. %  \
                        Round 1 Round 1             Round 1     Round 1   
                        Round 2 Round 2             Round 2     Round 2   
                        Round 3 Round 3             Round 3     Round 3   
                        Round 4 Round 4             Round 4     Round 4   
                        Round 5 Round 5             Round 5     Round 5   
0  Joshua Van Alexandre Pantoja    0  0  13 of 27  27 of 45    48%  60%   
1  Joshua Van Alexandre Pantoja    0  0  37 of 64  32 of 76    57%  42%   
2  Joshua Van Alexandre Pantoja    

In [15]:
def extrair_tabela_por_lutador(tabela_html):
    # Cabeçalhos das colunas
    headers = [th.get_text(strip=True) for th in tabela_html.find_all("th")]
    
    # Todas as linhas de dados (ignorando a linha de cabeçalho)
    linhas = tabela_html.find_all("tr")[1:]  # [1:] pula o <tr> do cabeçalho
    
    dados_lutador_1 = []
    dados_lutador_2 = []
    
    for linha in linhas:
        celulas = linha.find_all("td", class_="b-fight-details__table-col")
        valores_1 = []
        valores_2 = []
        for celula in celulas:
            ps = celula.find_all("p", class_="b-fight-details__table-text")
            valores_1.append(ps[0].get_text(strip=True) if len(ps) > 0 else None)
            valores_2.append(ps[1].get_text(strip=True) if len(ps) > 1 else None)
        dados_lutador_1.append(valores_1)
        dados_lutador_2.append(valores_2)
    
    return headers, dados_lutador_1, dados_lutador_2

In [16]:
tabelas_html = soup.find_all("table")
print(len(tabelas_html))  # deve dar 4, igual o pd.read_html

headers, l1, l2 = extrair_tabela_por_lutador(tabelas_html[0])
print(headers)
print(l1)
print(l2)

4
['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']
[['Joshua Van', '1', '181 of 305', '59%', '259 of 392', '0 of 0', '---', '1', '1', '2:13']]
[['Alexandre Pantoja', '0', '147 of 284', '51%', '159 of 300', '5 of 18', '27%', '0', '0', '9:11']]


In [17]:
df_totals = pd.DataFrame([l1[0], l2[0]], columns=headers)
df_totals

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl
0,Joshua Van,1,181 of 305,59%,259 of 392,0 of 0,---,1,1,2:13
1,Alexandre Pantoja,0,147 of 284,51%,159 of 300,5 of 18,27%,0,0,9:11


In [18]:
headers2, l1_2, l2_2 = extrair_tabela_por_lutador(tabelas_html[1])
print(headers2)
print(l1_2)
print(l2_2)

['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td %', 'Td %', 'Sub. att', 'Rev.', 'Ctrl', 'Round 1', 'Round 2', 'Round 3', 'Round 4', 'Round 5']
[[], ['Joshua Van', '0', '13 of 27', '48%', '46 of 61', '0 of 0', '---', '0', '1', '0:45'], [], ['Joshua Van', '0', '37 of 64', '57%', '48 of 76', '0 of 0', '---', '0', '0', '0:20'], [], ['Joshua Van', '0', '65 of 98', '66%', '67 of 101', '0 of 0', '---', '0', '0', '0:14'], [], ['Joshua Van', '0', '14 of 28', '50%', '41 of 57', '0 of 0', '---', '0', '0', '0:00'], [], ['Joshua Van', '1', '52 of 88', '59%', '57 of 97', '0 of 0', '---', '1', '0', '0:54']]
[[], ['Alexandre Pantoja', '0', '27 of 45', '60%', '30 of 48', '2 of 3', '66%', '0', '0', '2:25'], [], ['Alexandre Pantoja', '0', '32 of 76', '42%', '33 of 78', '1 of 5', '20%', '0', '0', '1:31'], [], ['Alexandre Pantoja', '0', '36 of 71', '50%', '37 of 72', '1 of 6', '16%', '0', '0', '1:17'], [], ['Alexandre Pantoja', '0', '14 of 22', '63%', '21 of 32', '1 of 2', '50%', '0', '0', 

In [19]:
def extrair_tabela_por_lutador(tabela_html):
    linhas = tabela_html.find_all("tr")
    
    # cabeçalhos: só da primeira linha (<tr>)
    headers = [th.get_text(strip=True) for th in linhas[0].find_all("th")]
    
    dados_lutador_1 = []
    dados_lutador_2 = []
    
    for linha in linhas[1:]:
        celulas = linha.find_all("td", class_="b-fight-details__table-col")
        if not celulas:
            continue  # pula linhas vazias/estruturais
        valores_1 = []
        valores_2 = []
        for celula in celulas:
            ps = celula.find_all("p", class_="b-fight-details__table-text")
            valores_1.append(ps[0].get_text(strip=True) if len(ps) > 0 else None)
            valores_2.append(ps[1].get_text(strip=True) if len(ps) > 1 else None)
        dados_lutador_1.append(valores_1)
        dados_lutador_2.append(valores_2)
    
    return headers, dados_lutador_1, dados_lutador_2

In [20]:
headers2, l1_2, l2_2 = extrair_tabela_por_lutador(tabelas_html[1])
print(headers2)
print(l1_2)
print(l2_2)

['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td %', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']
[['Joshua Van', '0', '13 of 27', '48%', '46 of 61', '0 of 0', '---', '0', '1', '0:45'], ['Joshua Van', '0', '37 of 64', '57%', '48 of 76', '0 of 0', '---', '0', '0', '0:20'], ['Joshua Van', '0', '65 of 98', '66%', '67 of 101', '0 of 0', '---', '0', '0', '0:14'], ['Joshua Van', '0', '14 of 28', '50%', '41 of 57', '0 of 0', '---', '0', '0', '0:00'], ['Joshua Van', '1', '52 of 88', '59%', '57 of 97', '0 of 0', '---', '1', '0', '0:54']]
[['Alexandre Pantoja', '0', '27 of 45', '60%', '30 of 48', '2 of 3', '66%', '0', '0', '2:25'], ['Alexandre Pantoja', '0', '32 of 76', '42%', '33 of 78', '1 of 5', '20%', '0', '0', '1:31'], ['Alexandre Pantoja', '0', '36 of 71', '50%', '37 of 72', '1 of 6', '16%', '0', '0', '1:17'], ['Alexandre Pantoja', '0', '14 of 22', '63%', '21 of 32', '1 of 2', '50%', '0', '0', '3:58'], ['Alexandre Pantoja', '0', '38 of 70', '54%', '38 of 70', '0 of 2', '0%', '0', '0'

In [21]:
headers_fixos = ['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']

df_por_round_l1 = pd.DataFrame(l1_2, columns=headers_fixos)
df_por_round_l1["Round"] = range(1, len(df_por_round_l1) + 1)

df_por_round_l2 = pd.DataFrame(l2_2, columns=headers_fixos)
df_por_round_l2["Round"] = range(1, len(df_por_round_l2) + 1)

df_por_round_l1

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Round
0,Joshua Van,0,13 of 27,48%,46 of 61,0 of 0,---,0,1,0:45,1
1,Joshua Van,0,37 of 64,57%,48 of 76,0 of 0,---,0,0,0:20,2
2,Joshua Van,0,65 of 98,66%,67 of 101,0 of 0,---,0,0,0:14,3
3,Joshua Van,0,14 of 28,50%,41 of 57,0 of 0,---,0,0,0:00,4
4,Joshua Van,1,52 of 88,59%,57 of 97,0 of 0,---,1,0,0:54,5


In [22]:
df_por_round_l1["Fighter"] = "Joshua Van"  # já vem null/errado da tabela, então sobrescrevemos
df_por_round_l1["Lutador_num"] = 1

df_por_round_l2["Fighter"] = "Alexandre Pantoja"
df_por_round_l2["Lutador_num"] = 2

In [23]:
df_por_round_l1["Fighter"] = "Joshua Van"
df_por_round_l1["Lutador_num"] = 1

df_por_round_l2["Fighter"] = "Alexandre Pantoja"
df_por_round_l2["Lutador_num"] = 2

df_por_round_l1  # <- essa linha final faz o Jupyter exibir o resultado

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Round,Lutador_num
0,Joshua Van,0,13 of 27,48%,46 of 61,0 of 0,---,0,1,0:45,1,1
1,Joshua Van,0,37 of 64,57%,48 of 76,0 of 0,---,0,0,0:20,2,1
2,Joshua Van,0,65 of 98,66%,67 of 101,0 of 0,---,0,0,0:14,3,1
3,Joshua Van,0,14 of 28,50%,41 of 57,0 of 0,---,0,0,0:00,4,1
4,Joshua Van,1,52 of 88,59%,57 of 97,0 of 0,---,1,0,0:54,5,1


In [24]:
headers3, l1_3, l2_3 = extrair_tabela_por_lutador(tabelas_html[2])
print(headers3)
print(l1_3)
print(l2_3)

['Fighter', 'Sig. str', 'Sig. str. %', 'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']
[['Joshua Van', '181 of 305', '59%', '158 of 278', '17 of 20', '6 of 7', '160 of 278', '17 of 21', '4 of 6']]
[['Alexandre Pantoja', '147 of 284', '51%', '91 of 213', '30 of 37', '26 of 34', '124 of 258', '16 of 18', '7 of 8']]


In [25]:
headers4, l1_4, l2_4 = extrair_tabela_por_lutador(tabelas_html[3])
print(headers4)
print(l1_4)
print(l2_4)

['Fighter', 'Sig. str', 'Sig. str. %', 'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']
[['Joshua Van', '13 of 27', '48%', '13 of 26', '0 of 1', '0 of 0', '9 of 22', '4 of 5', '0 of 0'], ['Joshua Van', '37 of 64', '57%', '27 of 52', '7 of 8', '3 of 4', '31 of 55', '6 of 9', '0 of 0'], ['Joshua Van', '65 of 98', '66%', '54 of 86', '9 of 10', '2 of 2', '58 of 91', '7 of 7', '0 of 0'], ['Joshua Van', '14 of 28', '50%', '13 of 27', '1 of 1', '0 of 0', '14 of 28', '0 of 0', '0 of 0'], ['Joshua Van', '52 of 88', '59%', '51 of 87', '0 of 0', '1 of 1', '48 of 82', '0 of 0', '4 of 6']]
[['Alexandre Pantoja', '27 of 45', '60%', '20 of 38', '4 of 4', '3 of 3', '18 of 35', '4 of 4', '5 of 6'], ['Alexandre Pantoja', '32 of 76', '42%', '21 of 60', '9 of 11', '2 of 5', '23 of 66', '8 of 9', '1 of 1'], ['Alexandre Pantoja', '36 of 71', '50%', '25 of 59', '7 of 8', '4 of 4', '33 of 67', '3 of 4', '0 of 0'], ['Alexandre Pantoja', '14 of 22', '63%', '11 of 19', '1 of 1', '2 of 2', '13 of 21', '0 of

In [26]:
nome_1, nome_2 = "Joshua Van", "Alexandre Pantoja"

headers_totais = ['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']
headers_sig = ['Fighter_sig', 'Sig. str_sig', 'Sig. str. %_sig', 'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']

df_resumo_l1 = pd.DataFrame([l1[0]], columns=headers_totais)
df_resumo_l1 = pd.concat([df_resumo_l1, pd.DataFrame([l1_3[0]], columns=headers_sig)], axis=1)
df_resumo_l1["Fighter"] = nome_1
df_resumo_l1 = df_resumo_l1.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])

df_resumo_l2 = pd.DataFrame([l2[0]], columns=headers_totais)
df_resumo_l2 = pd.concat([df_resumo_l2, pd.DataFrame([l2_3[0]], columns=headers_sig)], axis=1)
df_resumo_l2["Fighter"] = nome_2
df_resumo_l2 = df_resumo_l2.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])

df_resumo_luta = pd.concat([df_resumo_l1, df_resumo_l2], ignore_index=True)
df_resumo_luta

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Head,Body,Leg,Distance,Clinch,Ground
0,Joshua Van,1,181 of 305,59%,259 of 392,0 of 0,---,1,1,2:13,158 of 278,17 of 20,6 of 7,160 of 278,17 of 21,4 of 6
1,Alexandre Pantoja,0,147 of 284,51%,159 of 300,5 of 18,27%,0,0,9:11,91 of 213,30 of 37,26 of 34,124 of 258,16 of 18,7 of 8


In [27]:
headers_totais_round = ['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']
headers_sig_round = ['Fighter_sig', 'Sig. str_sig', 'Sig. str. %_sig', 'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']

# Lutador 1
df_round_l1 = pd.DataFrame(l1_2, columns=headers_totais_round)
df_round_l1 = pd.concat([df_round_l1, pd.DataFrame(l1_4, columns=headers_sig_round)], axis=1)
df_round_l1["Fighter"] = nome_1
df_round_l1["Round"] = range(1, len(df_round_l1) + 1)
df_round_l1 = df_round_l1.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])

# Lutador 2
df_round_l2 = pd.DataFrame(l2_2, columns=headers_totais_round)
df_round_l2 = pd.concat([df_round_l2, pd.DataFrame(l2_4, columns=headers_sig_round)], axis=1)
df_round_l2["Fighter"] = nome_2
df_round_l2["Round"] = range(1, len(df_round_l2) + 1)
df_round_l2 = df_round_l2.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])

df_por_round = pd.concat([df_round_l1, df_round_l2], ignore_index=True)
df_por_round

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Head,Body,Leg,Distance,Clinch,Ground,Round
0,Joshua Van,0,13 of 27,48%,46 of 61,0 of 0,---,0,1,0:45,13 of 26,0 of 1,0 of 0,9 of 22,4 of 5,0 of 0,1
1,Joshua Van,0,37 of 64,57%,48 of 76,0 of 0,---,0,0,0:20,27 of 52,7 of 8,3 of 4,31 of 55,6 of 9,0 of 0,2
2,Joshua Van,0,65 of 98,66%,67 of 101,0 of 0,---,0,0,0:14,54 of 86,9 of 10,2 of 2,58 of 91,7 of 7,0 of 0,3
3,Joshua Van,0,14 of 28,50%,41 of 57,0 of 0,---,0,0,0:00,13 of 27,1 of 1,0 of 0,14 of 28,0 of 0,0 of 0,4
4,Joshua Van,1,52 of 88,59%,57 of 97,0 of 0,---,1,0,0:54,51 of 87,0 of 0,1 of 1,48 of 82,0 of 0,4 of 6,5
5,Alexandre Pantoja,0,27 of 45,60%,30 of 48,2 of 3,66%,0,0,2:25,20 of 38,4 of 4,3 of 3,18 of 35,4 of 4,5 of 6,1
6,Alexandre Pantoja,0,32 of 76,42%,33 of 78,1 of 5,20%,0,0,1:31,21 of 60,9 of 11,2 of 5,23 of 66,8 of 9,1 of 1,2
7,Alexandre Pantoja,0,36 of 71,50%,37 of 72,1 of 6,16%,0,0,1:17,25 of 59,7 of 8,4 of 4,33 of 67,3 of 4,0 of 0,3
8,Alexandre Pantoja,0,14 of 22,63%,21 of 32,1 of 2,50%,0,0,3:58,11 of 19,1 of 1,2 of 2,13 of 21,0 of 0,1 of 1,4
9,Alexandre Pantoja,0,38 of 70,54%,38 of 70,0 of 2,0%,0,0,0:00,14 of 37,9 of 13,15 of 20,37 of 69,1 of 1,0 of 0,5


In [28]:
df_resumo_luta["Method"] = "Decision - Unanimous"  # ajuste conforme o que veio do Passo 9
df_resumo_luta["Final_Round"] = 5
df_resumo_luta["Time"] = "5:00"
df_resumo_luta["Referee"] = "Herb Dean"
df_resumo_luta["Fight_URL"] = url

df_por_round["Fight_URL"] = url

In [29]:
df_resumo_luta.to_csv("../data/raw/resumo_luta_teste.csv", index=False)
df_por_round.to_csv("../data/raw/por_round_luta_teste.csv", index=False)

print("Salvo com sucesso!")

Salvo com sucesso!


In [30]:
def extrair_tabela_por_lutador(tabela_html):
    linhas = tabela_html.find_all("tr")
    headers = [th.get_text(strip=True) for th in linhas[0].find_all("th")]
    
    dados_lutador_1 = []
    dados_lutador_2 = []
    
    for linha in linhas[1:]:
        celulas = linha.find_all("td", class_="b-fight-details__table-col")
        if not celulas:
            continue
        valores_1 = []
        valores_2 = []
        for celula in celulas:
            ps = celula.find_all("p", class_="b-fight-details__table-text")
            valores_1.append(ps[0].get_text(strip=True) if len(ps) > 0 else None)
            valores_2.append(ps[1].get_text(strip=True) if len(ps) > 1 else None)
        dados_lutador_1.append(valores_1)
        dados_lutador_2.append(valores_2)
    
    return headers, dados_lutador_1, dados_lutador_2

In [31]:
def extrair_dados_luta(url, driver):
    # Baixar e renderizar a página
    driver.get(url)
    time.sleep(5)
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    
    # Nomes e resultados (W/L)
    pessoas = soup.find_all("div", class_="b-fight-details__person")
    nome_1 = pessoas[0].find("a", class_="b-link b-fight-details__person-link").text.strip()
    resultado_1 = pessoas[0].find("i", class_="b-fight-details__person-status").text.strip()
    nome_2 = pessoas[1].find("a", class_="b-link b-fight-details__person-link").text.strip()
    resultado_2 = pessoas[1].find("i", class_="b-fight-details__person-status").text.strip()
    
    # Método, round final, tempo, referee
    info_bloco = soup.find("p", class_="b-fight-details__text")
    itens = info_bloco.find_all("i", class_=["b-fight-details__text-item_first", "b-fight-details__text-item"])
    info = {}
    for item in itens:
        label = item.find("i", class_="b-fight-details__label").text.strip()
        valor = item.get_text().replace(label, "").strip()
        info[label] = valor
    
    metodo = info.get("Method:", None)
    round_final = info.get("Round:", None)
    tempo = info.get("Time:", None)
    referee = info.get("Referee:", None)
    
    # Tabelas
    tabelas_html = soup.find_all("table")
    headers0, l1_0, l2_0 = extrair_tabela_por_lutador(tabelas_html[0])
    headers1, l1_1, l2_1 = extrair_tabela_por_lutador(tabelas_html[1])
    headers2, l1_2, l2_2 = extrair_tabela_por_lutador(tabelas_html[2])
    headers3, l1_3, l2_3 = extrair_tabela_por_lutador(tabelas_html[3])
    
    headers_totais = ['Fighter', 'KD', 'Sig. str.', 'Sig. str. %', 'Total str.', 'Td', 'Td %', 'Sub. att', 'Rev.', 'Ctrl']
    headers_sig = ['Fighter_sig', 'Sig. str_sig', 'Sig. str. %_sig', 'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']
    
    # Resumo da luta
    df_resumo_l1 = pd.DataFrame([l1_0[0]], columns=headers_totais)
    df_resumo_l1 = pd.concat([df_resumo_l1, pd.DataFrame([l1_2[0]], columns=headers_sig)], axis=1)
    df_resumo_l1["Fighter"] = nome_1
    df_resumo_l1["Resultado"] = resultado_1
    df_resumo_l1 = df_resumo_l1.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])
    
    df_resumo_l2 = pd.DataFrame([l2_0[0]], columns=headers_totais)
    df_resumo_l2 = pd.concat([df_resumo_l2, pd.DataFrame([l2_2[0]], columns=headers_sig)], axis=1)
    df_resumo_l2["Fighter"] = nome_2
    df_resumo_l2["Resultado"] = resultado_2
    df_resumo_l2 = df_resumo_l2.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])
    
    df_resumo_luta = pd.concat([df_resumo_l1, df_resumo_l2], ignore_index=True)
    df_resumo_luta["Method"] = metodo
    df_resumo_luta["Final_Round"] = round_final
    df_resumo_luta["Time"] = tempo
    df_resumo_luta["Referee"] = referee
    df_resumo_luta["Fight_URL"] = url
    
    # Por round
    df_round_l1 = pd.DataFrame(l1_1, columns=headers_totais)
    df_round_l1 = pd.concat([df_round_l1, pd.DataFrame(l1_3, columns=headers_sig)], axis=1)
    df_round_l1["Fighter"] = nome_1
    df_round_l1["Round"] = range(1, len(df_round_l1) + 1)
    df_round_l1 = df_round_l1.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])
    
    df_round_l2 = pd.DataFrame(l2_1, columns=headers_totais)
    df_round_l2 = pd.concat([df_round_l2, pd.DataFrame(l2_3, columns=headers_sig)], axis=1)
    df_round_l2["Fighter"] = nome_2
    df_round_l2["Round"] = range(1, len(df_round_l2) + 1)
    df_round_l2 = df_round_l2.drop(columns=["Fighter_sig", "Sig. str_sig", "Sig. str. %_sig"])
    
    df_por_round = pd.concat([df_round_l1, df_round_l2], ignore_index=True)
    df_por_round["Fight_URL"] = url
    
    return df_resumo_luta, df_por_round

In [32]:
df_resumo_teste, df_round_teste = extrair_dados_luta(
    "http://ufcstats.com/fight-details/568ec6af4008355a",
    driver
)

df_resumo_teste

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,...,Leg,Distance,Clinch,Ground,Resultado,Method,Final_Round,Time,Referee,Fight_URL
0,Joshua Van,1,181 of 305,59%,259 of 392,0 of 0,---,1,1,2:13,...,6 of 7,160 of 278,17 of 21,4 of 6,W,Decision - Unanimous,5,5:00,Herb Dean,http://ufcstats.com/fight-details/568ec6af4008...
1,Alexandre Pantoja,0,147 of 284,51%,159 of 300,5 of 18,27%,0,0,9:11,...,26 of 34,124 of 258,16 of 18,7 of 8,L,Decision - Unanimous,5,5:00,Herb Dean,http://ufcstats.com/fight-details/568ec6af4008...


In [33]:
df_round_teste

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Head,Body,Leg,Distance,Clinch,Ground,Round,Fight_URL
0,Joshua Van,0,13 of 27,48%,46 of 61,0 of 0,---,0,1,0:45,13 of 26,0 of 1,0 of 0,9 of 22,4 of 5,0 of 0,1,http://ufcstats.com/fight-details/568ec6af4008...
1,Joshua Van,0,37 of 64,57%,48 of 76,0 of 0,---,0,0,0:20,27 of 52,7 of 8,3 of 4,31 of 55,6 of 9,0 of 0,2,http://ufcstats.com/fight-details/568ec6af4008...
2,Joshua Van,0,65 of 98,66%,67 of 101,0 of 0,---,0,0,0:14,54 of 86,9 of 10,2 of 2,58 of 91,7 of 7,0 of 0,3,http://ufcstats.com/fight-details/568ec6af4008...
3,Joshua Van,0,14 of 28,50%,41 of 57,0 of 0,---,0,0,0:00,13 of 27,1 of 1,0 of 0,14 of 28,0 of 0,0 of 0,4,http://ufcstats.com/fight-details/568ec6af4008...
4,Joshua Van,1,52 of 88,59%,57 of 97,0 of 0,---,1,0,0:54,51 of 87,0 of 0,1 of 1,48 of 82,0 of 0,4 of 6,5,http://ufcstats.com/fight-details/568ec6af4008...
5,Alexandre Pantoja,0,27 of 45,60%,30 of 48,2 of 3,66%,0,0,2:25,20 of 38,4 of 4,3 of 3,18 of 35,4 of 4,5 of 6,1,http://ufcstats.com/fight-details/568ec6af4008...
6,Alexandre Pantoja,0,32 of 76,42%,33 of 78,1 of 5,20%,0,0,1:31,21 of 60,9 of 11,2 of 5,23 of 66,8 of 9,1 of 1,2,http://ufcstats.com/fight-details/568ec6af4008...
7,Alexandre Pantoja,0,36 of 71,50%,37 of 72,1 of 6,16%,0,0,1:17,25 of 59,7 of 8,4 of 4,33 of 67,3 of 4,0 of 0,3,http://ufcstats.com/fight-details/568ec6af4008...
8,Alexandre Pantoja,0,14 of 22,63%,21 of 32,1 of 2,50%,0,0,3:58,11 of 19,1 of 1,2 of 2,13 of 21,0 of 0,1 of 1,4,http://ufcstats.com/fight-details/568ec6af4008...
9,Alexandre Pantoja,0,38 of 70,54%,38 of 70,0 of 2,0%,0,0,0:00,14 of 37,9 of 13,15 of 20,37 of 69,1 of 1,0 of 0,5,http://ufcstats.com/fight-details/568ec6af4008...


In [35]:
df_resumo_teste2, df_round_teste2 = extrair_dados_luta(
    "http://ufcstats.com/fight-details/034c04e480d87b8f",
    driver
)

df_resumo_teste2

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,...,Leg,Distance,Clinch,Ground,Resultado,Method,Final_Round,Time,Referee,Fight_URL
0,Jean Silva,1,39 of 77,50%,40 of 78,1 of 2,50%,1,0,1:37,...,2 of 2,32 of 66,3 of 5,4 of 6,W,Submission,3,2:57,Dan Miragliotta,http://ufcstats.com/fight-details/034c04e480d8...
1,Jose Delgado,0,40 of 108,37%,41 of 109,0 of 1,0%,0,0,0:21,...,18 of 22,35 of 98,5 of 9,0 of 1,L,Submission,3,2:57,Dan Miragliotta,http://ufcstats.com/fight-details/034c04e480d8...


In [36]:
df_round_teste2

,Fighter,KD,Sig. str.,Sig. str. %,Total str.,Td,Td %,Sub. att,Rev.,Ctrl,Head,Body,Leg,Distance,Clinch,Ground,Round,Fight_URL
0,Jean Silva,0,3 of 5,60%,3 of 5,0 of 1,0%,0,0,0:11,2 of 4,1 of 1,0 of 0,3 of 5,0 of 0,0 of 0,1,http://ufcstats.com/fight-details/034c04e480d8...
1,Jean Silva,0,23 of 50,46%,24 of 51,1 of 1,100%,0,0,1:02,18 of 40,5 of 10,0 of 0,17 of 40,3 of 5,3 of 5,2,http://ufcstats.com/fight-details/034c04e480d8...
2,Jean Silva,1,13 of 22,59%,13 of 22,0 of 0,---,1,0,0:24,7 of 16,4 of 4,2 of 2,12 of 21,0 of 0,1 of 1,3,http://ufcstats.com/fight-details/034c04e480d8...
3,Jose Delgado,0,17 of 34,50%,18 of 35,0 of 0,---,0,0,0:21,4 of 16,2 of 4,11 of 14,16 of 32,1 of 2,0 of 0,1,http://ufcstats.com/fight-details/034c04e480d8...
4,Jose Delgado,0,14 of 48,29%,14 of 48,0 of 0,---,0,0,0:00,8 of 42,3 of 3,3 of 3,10 of 41,4 of 7,0 of 0,2,http://ufcstats.com/fight-details/034c04e480d8...
5,Jose Delgado,0,9 of 26,34%,9 of 26,0 of 1,0%,0,0,0:00,5 of 21,0 of 0,4 of 5,9 of 25,0 of 0,0 of 1,3,http://ufcstats.com/fight-details/034c04e480d8...


In [38]:
url_evento = "http://ufcstats.com/event-details/b96619b3acd7d9da"
driver.get(url_evento)
time.sleep(5)

html_evento = driver.page_source
soup_evento = BeautifulSoup(html_evento, "html.parser")

linhas_luta = soup_evento.find_all("tr", class_="b-fight-details__table-row")
urls_lutas = [linha["data-link"] for linha in linhas_luta if linha.has_attr("data-link")]

print(len(urls_lutas))
print(urls_lutas)

12
['http://ufcstats.com/fight-details/365fd759c03e93cf', 'http://ufcstats.com/fight-details/4700d42c9869674a', 'http://ufcstats.com/fight-details/d4bd456d70965734', 'http://ufcstats.com/fight-details/53902dcb3e213b18', 'http://ufcstats.com/fight-details/8e482cead0c7eb15', 'http://ufcstats.com/fight-details/6e495deac1f57169', 'http://ufcstats.com/fight-details/e7c11d1b22d50ba0', 'http://ufcstats.com/fight-details/9c5da658c76da7c0', 'http://ufcstats.com/fight-details/e79e4d13546d2360', 'http://ufcstats.com/fight-details/360b13f97e7ca703', 'http://ufcstats.com/fight-details/e7077d030d5fc15f', 'http://ufcstats.com/fight-details/d5ebdf462c9b59b3']


In [39]:
todos_resumos = []
todos_rounds = []

for i, url_luta in enumerate(urls_lutas):
    print(f"Processando luta {i+1}/{len(urls_lutas)}: {url_luta}")
    try:
        df_resumo, df_round = extrair_dados_luta(url_luta, driver)
        todos_resumos.append(df_resumo)
        todos_rounds.append(df_round)
    except Exception as e:
        print(f"  Erro nessa luta: {e}")

df_evento_resumo = pd.concat(todos_resumos, ignore_index=True)
df_evento_round = pd.concat(todos_rounds, ignore_index=True)

print("Concluído!")
print(df_evento_resumo.shape, df_evento_round.shape)

Processando luta 1/12: http://ufcstats.com/fight-details/365fd759c03e93cf
Processando luta 2/12: http://ufcstats.com/fight-details/4700d42c9869674a
Processando luta 3/12: http://ufcstats.com/fight-details/d4bd456d70965734
Processando luta 4/12: http://ufcstats.com/fight-details/53902dcb3e213b18
Processando luta 5/12: http://ufcstats.com/fight-details/8e482cead0c7eb15
Processando luta 6/12: http://ufcstats.com/fight-details/6e495deac1f57169
Processando luta 7/12: http://ufcstats.com/fight-details/e7c11d1b22d50ba0
Processando luta 8/12: http://ufcstats.com/fight-details/9c5da658c76da7c0
Processando luta 9/12: http://ufcstats.com/fight-details/e79e4d13546d2360
Processando luta 10/12: http://ufcstats.com/fight-details/360b13f97e7ca703
Processando luta 11/12: http://ufcstats.com/fight-details/e7077d030d5fc15f
Processando luta 12/12: http://ufcstats.com/fight-details/d5ebdf462c9b59b3
Concluído!
(24, 22) (66, 18)


In [40]:
df_evento_resumo["Event_URL"] = url_evento
df_evento_round["Event_URL"] = url_evento

df_evento_resumo.to_csv("../data/raw/evento_teste_resumo.csv", index=False)
df_evento_round.to_csv("../data/raw/evento_teste_round.csv", index=False)

print("Evento salvo com sucesso!")

Evento salvo com sucesso!


In [41]:
def extrair_urls_evento(url_evento, driver):
    driver.get(url_evento)
    time.sleep(5)
    html_evento = driver.page_source
    soup_evento = BeautifulSoup(html_evento, "html.parser")
    
    linhas_luta = soup_evento.find_all("tr", class_="b-fight-details__table-row")
    urls_lutas = [linha["data-link"] for linha in linhas_luta if linha.has_attr("data-link")]
    
    return urls_lutas

In [42]:
def extrair_evento(url_evento, driver):
    urls_lutas = extrair_urls_evento(url_evento, driver)
    
    todos_resumos = []
    todos_rounds = []
    
    for i, url_luta in enumerate(urls_lutas):
        print(f"  Luta {i+1}/{len(urls_lutas)}: {url_luta}")
        try:
            df_resumo, df_round = extrair_dados_luta(url_luta, driver)
            todos_resumos.append(df_resumo)
            todos_rounds.append(df_round)
        except Exception as e:
            print(f"    Erro nessa luta: {e}")
    
    if not todos_resumos:
        return None, None
    
    df_evento_resumo = pd.concat(todos_resumos, ignore_index=True)
    df_evento_round = pd.concat(todos_rounds, ignore_index=True)
    
    df_evento_resumo["Event_URL"] = url_evento
    df_evento_round["Event_URL"] = url_evento
    
    return df_evento_resumo, df_evento_round

In [43]:
url_evento2 = "http://ufcstats.com/event-details/2144954270be834d"
df_evento2_resumo, df_evento2_round = extrair_evento(url_evento2, driver)

print(df_evento2_resumo.shape if df_evento2_resumo is not None else "Falhou")
print(df_evento2_round.shape if df_evento2_round is not None else "Falhou")

  Luta 1/14: http://ufcstats.com/fight-details/bba8c0ab999b9e6f
  Luta 2/14: http://ufcstats.com/fight-details/55f698e5c4ed2c47
  Luta 3/14: http://ufcstats.com/fight-details/c23197c7e4f2fd9d
  Luta 4/14: http://ufcstats.com/fight-details/8300a095a4908ab1
  Luta 5/14: http://ufcstats.com/fight-details/cbf9063d23513661
  Luta 6/14: http://ufcstats.com/fight-details/b90b099c1f47bb93
  Luta 7/14: http://ufcstats.com/fight-details/ad17c34e0cc7d702
  Luta 8/14: http://ufcstats.com/fight-details/b1c6ea7d7d413bd5
  Luta 9/14: http://ufcstats.com/fight-details/94d411112ebe8fad
  Luta 10/14: http://ufcstats.com/fight-details/ac3af13ffe39b1d8
  Luta 11/14: http://ufcstats.com/fight-details/1f973e21581b082a
  Luta 12/14: http://ufcstats.com/fight-details/809ebb0cfdd01ce2
  Luta 13/14: http://ufcstats.com/fight-details/e5fd7ef7ccf955b8
  Luta 14/14: http://ufcstats.com/fight-details/9226a13d001b49ec
(28, 23)
(62, 19)


In [44]:
url_lista_eventos = "http://ufcstats.com/statistics/events/completed"
driver.get(url_lista_eventos)
time.sleep(5)

html_lista = driver.page_source
soup_lista = BeautifulSoup(html_lista, "html.parser")

links_evento = soup_lista.find_all("a", class_="b-link b-link_style_black")

eventos = []
for link in links_evento:
    nome = link.text.strip()
    url = link["href"]
    eventos.append({"nome": nome, "url": url})

print(len(eventos))
print(eventos[:5])  # mostra os 5 primeiros, pra conferir

24
[{'nome': 'UFC 331: Van vs. Pantoja 2', 'url': 'http://ufcstats.com/event-details/8a0a35e7c74bebcc'}, {'nome': 'Noche UFC: Silva vs. Delgado', 'url': 'http://ufcstats.com/event-details/638cfec7ec559d6e'}, {'nome': 'UFC Fight Night: Hooker vs. Parnasse', 'url': 'http://ufcstats.com/event-details/2144954270be834d'}, {'nome': 'UFC Fight Night: Nurmagomedov vs. Song', 'url': 'http://ufcstats.com/event-details/9d61d8cb1c354867'}, {'nome': 'UFC Fight Night: Hernandez vs. Rodrigues', 'url': 'http://ufcstats.com/event-details/a0a69dc9914ef6e1'}]


In [45]:
def extrair_varios_eventos(eventos, driver, limite=None):
    lista_eventos = eventos[:limite] if limite else eventos
    
    todos_resumos = []
    todos_rounds = []
    
    for i, evento in enumerate(lista_eventos):
        print(f"=== Evento {i+1}/{len(lista_eventos)}: {evento['nome']} ===")
        try:
            df_resumo, df_round = extrair_evento(evento["url"], driver)
            if df_resumo is not None:
                df_resumo["Event_Name"] = evento["nome"]
                df_round["Event_Name"] = evento["nome"]
                todos_resumos.append(df_resumo)
                todos_rounds.append(df_round)
        except Exception as e:
            print(f"  Erro nesse evento: {e}")
        time.sleep(2)  # pequena pausa entre eventos
    
    df_final_resumo = pd.concat(todos_resumos, ignore_index=True)
    df_final_round = pd.concat(todos_rounds, ignore_index=True)
    
    return df_final_resumo, df_final_round

In [46]:
df_dataset_resumo, df_dataset_round = extrair_varios_eventos(eventos, driver, limite=3)

print(df_dataset_resumo.shape)
print(df_dataset_round.shape)

=== Evento 1/3: UFC 331: Van vs. Pantoja 2 ===
  Luta 1/12: http://ufcstats.com/fight-details/568ec6af4008355a
  Luta 2/12: http://ufcstats.com/fight-details/fe612d20eec7ef35
  Luta 3/12: http://ufcstats.com/fight-details/f7810c98786c3dd8
  Luta 4/12: http://ufcstats.com/fight-details/023ce4fa7b7b76f9
  Luta 5/12: http://ufcstats.com/fight-details/196c64693a2ce308
  Luta 6/12: http://ufcstats.com/fight-details/d0361fe28d05a0b5
  Luta 7/12: http://ufcstats.com/fight-details/3f2d637ffb324a51
  Luta 8/12: http://ufcstats.com/fight-details/d89cc2cb92cd8407
  Luta 9/12: http://ufcstats.com/fight-details/51789f77811b1817
  Luta 10/12: http://ufcstats.com/fight-details/9be47241c960d082
  Luta 11/12: http://ufcstats.com/fight-details/73304588cef4f88b
  Luta 12/12: http://ufcstats.com/fight-details/2e35163040eb3189
=== Evento 2/3: Noche UFC: Silva vs. Delgado ===
  Luta 1/13: http://ufcstats.com/fight-details/034c04e480d87b8f
  Luta 2/13: http://ufcstats.com/fight-details/1aa008d8597a73b8
  Lut

In [47]:
df_dataset_resumo.to_csv("../data/raw/dataset_3eventos_resumo.csv", index=False)
df_dataset_round.to_csv("../data/raw/dataset_3eventos_round.csv", index=False)

print("Dataset de 3 eventos salvo com sucesso!")

Dataset de 3 eventos salvo com sucesso!
